# BioSample RO-Crate overview

This notebook inspects the raw and curated contents of the BioSample RO-Crate produced by `bsllmner-mk2`. It describes the data categories, compares raw inputs with curated outputs, examines overlap between ChIP-Atlas and DDBJ RNA-Seq, and shows one paired example from each category. It does not evaluate the trace-back pipeline; that's in `biosample_trace_back_analysis.ipynb` and `biosample_trace_back_evaluation.ipynb`.

In [1]:
# TABLE OF CONTENTS
import sys

sys.path.insert(0, "../src")
from notebook_utils import table_of_contents

table_of_contents("biosample_rocrate_overview.ipynb")

### Table of contents

- [Setup](#Setup)
- [What's in the raw input and the curated output](#What's-in-the-raw-input-and-the-curated-output)
- [Distribution across the four categories](#Distribution-across-the-four-categories)
- [Overlap between ChIP-Atlas and DDBJ RNA-Seq](#Overlap-between-ChIP-Atlas-and-DDBJ-RNA-Seq)
- [One paired example per category](#One-paired-example-per-category)

## Setup

In [2]:
# IMPORTS
import json
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from bs_entries import (
    MATCH_STRATEGIES,
    bold_span_html,
    bs_record_table,
    find_bs_entry,
    find_bs_entry_in_files,
    get_accession,
    iter_bs_entries,
    matches_display_table,
    scan_accessions,
    verify_extracted_against_raw,
    verify_extracted_against_raw_rows,
)
from llm_evidence import verify_not_found_with_llm
from notebook_utils import display_with_nested_tables, md, table_covers_json
from ontology import build_field_ontology_indexes
from paths import (
    CRATE_CONFIG_DIR,
    CRATE_INPUTS_DIR,
    CRATE_ONTOLOGY_DIR,
    CRATE_RESULTS_DIR,
    DERIVED_DIR,
    MANUAL_REVIEW_SAMPLE_PARQUET,
    RUN_INDEX_TSV,
    TRACE_BACK_FULL_PARQUET,
)
from select_results import curated_entry_table, find_entry, load_select_result

/workspace/BH26/BH26_BioSample_curation/.conda_env/lib/python3.12/site-packages/torch/cuda/__init__.py:228: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Each category is a group of runs in `provenance/run_index.tsv` (one row per run = one input file -> one result file), identified by matching the start of the `dataset` column.

In [3]:
# THE FOUR CATEGORIES, AS A FILTER OVER run_index.tsv's `dataset` COLUMN
@dataclass(frozen=True)
class Category:
    name: str
    organism: str
    technology: str
    dataset_prefix: str  # matches run_index.tsv's `dataset` column by startswith


CATEGORIES = [
    Category("DDBJ RNA-Seq", "Homo sapiens", "RNA-Seq", "rnaseq_human"),
    Category("DDBJ RNA-Seq", "Mus musculus", "RNA-Seq", "rnaseq_mouse"),
    Category("ChIP-Atlas", "Homo sapiens", "ChIP-Atlas (hg38)", "chipatlas_hg38"),
    Category("ChIP-Atlas", "Mus musculus", "ChIP-Atlas (mm10)", "chipatlas_mm10"),
]

run_index = pd.read_csv(RUN_INDEX_TSV, sep="\t")


def category_input_files(category: Category) -> list[Path]:
    datasets = sorted(run_index.loc[run_index["dataset"].str.startswith(category.dataset_prefix), "dataset"].unique())
    files: list[Path] = []
    for dataset in datasets:
        files.extend(sorted((CRATE_INPUTS_DIR / dataset).glob("bs_entries_*.jsonl")))
    return files

In [4]:
# ONTOLOGY INDEXES, BUILT ONCE PER SELECT CONFIG AND REUSED EVERYWHERE THEY'RE NEEDED
_ontology_indexes_cache: dict[str, dict] = {}


def get_ontology_indexes(select_config_filename: str) -> dict:
    if select_config_filename not in _ontology_indexes_cache:
        with (CRATE_CONFIG_DIR / select_config_filename).open() as f:
            select_config = json.load(f)
        fields = list(select_config.get("fields", {}).keys())
        _ontology_indexes_cache[select_config_filename] = build_field_ontology_indexes(select_config, CRATE_ONTOLOGY_DIR, fields)
    return _ontology_indexes_cache[select_config_filename]

Every raw/curated record below is shown as a readable table (built by `bs_record_table`/`curated_entry_table` -- every field, including empty ones, nothing filtered out). The tables are verified once, live, against the underlying JSON (`notebook_utils.table_covers_json`) to confirm they actually reproduce every value -- the JSON itself is never printed, since the table already covers it.

In [5]:
# SHOW A RECORD AS A TABLE, WITH A ONE-TIME LIVE COMPLETENESS CHECK AGAINST THE JSON
checked_complete = {"raw": False, "curated": False}


def show_raw_record(raw_record: dict, label: str) -> None:
    table = bs_record_table(raw_record)
    if not checked_complete["raw"]:
        covers = table_covers_json(raw_record, table)
        md(f"(Live-verified: this table format reproduces every value in the raw JSON: **{covers}**.)")
        checked_complete["raw"] = True
    md(f"**{label}:**")
    display_with_nested_tables(table)


def show_curated_record(entry: dict, label: str) -> None:
    table = curated_entry_table(entry)
    md(f"**{label}:**")
    display_with_nested_tables(table)
    if not checked_complete["curated"]:
        trimmed = {"extracted": entry["extract"]["extracted"], "results": entry["results"]}
        covers = table_covers_json(trimmed, table)
        md(f"(Live-verified: this table format reproduces every value in the curated JSON: **{covers}**.)")
        checked_complete["curated"] = True

## What's in the raw input and the curated output

Every input file under `inputs/` is a JSONL of raw NCBI/DDBJ/EBI BioSample records (submitter title, organism, free-text/structured attributes). Every result file under `results/` is a `SelectResult`: for each input record (matched by its `accession`), the values an LLM extracted from it (`extract.extracted`) and the ontology terms those values were mapped to (`results`). That `accession` match is the only thing pairing one to the other -- there's no positional/line-number link.

## Distribution across the four categories

Record and run counts per category, computed live from `run_index.tsv`:

In [6]:
# RECORD/RUN COUNTS PER CATEGORY
def category_counts(categories: list[Category]) -> pd.DataFrame:
    rows = []
    for cat in categories:
        subset = run_index[run_index["dataset"].str.startswith(cat.dataset_prefix)]
        rows.append(
            {
                "category": cat.name,
                "organism": cat.organism,
                "technology": cat.technology,
                "n_runs": len(subset),
                "n_records": int(subset["total_input_entries"].sum()),
            }
        )
    counts = pd.DataFrame(rows)
    counts["n_records"] = counts["n_records"].map("{:,}".format)
    return counts


category_counts(CATEGORIES)

,category,organism,technology,n_runs,n_records
0,DDBJ RNA-Seq,Homo sapiens,RNA-Seq,148,"2,111,841"
1,DDBJ RNA-Seq,Mus musculus,RNA-Seq,149,"1,747,288"
2,ChIP-Atlas,Homo sapiens,ChIP-Atlas (hg38),7,"179,015"
3,ChIP-Atlas,Mus musculus,ChIP-Atlas (mm10),7,"188,122"


`total_input_entries` sums to the number of raw BioSample records available; the DDBJ RNA-Seq split is assay-pure by construction, but ChIP-Atlas is not -- it indexes ChIP-seq, ATAC-Seq, DNase-Seq and more under one crawl, and this crate's local files don't carry a per-record assay label for it (that classification lives in ChIP-Atlas's own metadata, outside this crate), so the "technology" column above is a single collapsed label for the whole ChIP-Atlas crawl rather than a per-assay breakdown.

## Overlap between ChIP-Atlas and DDBJ RNA-Seq

ChIP-Atlas and DDBJ RNA-Seq are two separate crawls over BioSample, not disjoint-by-construction data sources -- so the same physical BioSample accession can in principle appear in both. Checked directly by comparing the full accession sets from each source's input files, per organism:

In [7]:
# ACCESSION-SET OVERLAP BETWEEN ChIP-Atlas AND DDBJ RNA-Seq, PER ORGANISM
accession_sets = {cat.dataset_prefix: scan_accessions(category_input_files(cat)) for cat in CATEGORIES}


def overlap_row(organism: str, chip_prefix: str, ddbj_prefix: str) -> dict:
    chip_set = accession_sets[chip_prefix]
    ddbj_set = accession_sets[ddbj_prefix]
    overlap = chip_set & ddbj_set
    return {
        "organism": organism,
        "chip_atlas_n": f"{len(chip_set):,}",
        "ddbj_rnaseq_n": f"{len(ddbj_set):,}",
        "overlap_n": f"{len(overlap):,}",
        "pct_of_chip_atlas": round(100 * len(overlap) / len(chip_set), 2),
        "pct_of_ddbj_rnaseq": round(100 * len(overlap) / len(ddbj_set), 2),
    }


overlap_table = pd.DataFrame(
    [
        overlap_row("Homo sapiens", "chipatlas_hg38", "rnaseq_human"),
        overlap_row("Mus musculus", "chipatlas_mm10", "rnaseq_mouse"),
    ]
)
overlap_table

,organism,chip_atlas_n,ddbj_rnaseq_n,overlap_n,pct_of_chip_atlas,pct_of_ddbj_rnaseq
0,Homo sapiens,"179,015","2,097,428",899,0.50,0.04
1,Mus musculus,"188,122","1,724,474",432,0.23,0.03


So the two sources are mostly disjoint but not entirely: a small fraction of accessions belong to both. Manually checking a few of the overlapping accessions' titles turns up two plausible explanations, and this crate's local data can't fully distinguish between them: some are multi-assay submissions (e.g. ENCODE biosamples, where one donor/tissue-prep BioSample record is shared by several different sequencing experiments -- ChIP-seq, RNA-seq, etc. -- against it), and ChIP-Atlas's own classification includes non-ChIP-seq track types (it also indexes RNA-Seq, ATAC-Seq, DNase-Seq, etc., a per-record label this crate's local files don't carry -- that lives in ChIP-Atlas's own `experimentList.tab`, external to this crate). This is not a gap in the pipeline: DDBJ RNA-Seq was deliberately scoped to RNA-Seq-assay BioSamples only, and most ChIP-Atlas accessions are not RNA-Seq experiments to begin with, so there's no reason to expect them to show up in the RNA-Seq crawl at all. The tiny overlap that does exist represents genuine dual-use samples, not something the pipeline "missed."

One overlapping accession per organism, both raw copies merged into one table (field, ChIP-Atlas value, DDBJ RNA-Seq value) so they're easy to compare directly -- any scalar field where the two sources disagree is **bolded** on both sides:

In [8]:
# ONE ACCESSION PRESENT IN BOTH SOURCES, PER ORGANISM -- MERGED INTO ONE COMPARISON TABLE
def merged_overlap_table(chip_record: dict, ddbj_record: dict) -> pd.DataFrame:
    chip_row = bs_record_table(chip_record).iloc[0]
    ddbj_row = bs_record_table(ddbj_record).iloc[0]

    rows = []
    for field in chip_row.index:
        chip_val, ddbj_val = chip_row[field], ddbj_row[field]
        if isinstance(chip_val, pd.DataFrame) or isinstance(ddbj_val, pd.DataFrame):
            # nested (attributes/ids/links) -- shown as-is, side by side, rather than diffed row-by-row
            rows.append({"field": field, "ChIP-Atlas": chip_val, "DDBJ RNA-Seq": ddbj_val})
            continue
        differs = str(chip_val) != str(ddbj_val)
        if differs:
            chip_val = bold_span_html(str(chip_val), (0, len(str(chip_val))))
            ddbj_val = bold_span_html(str(ddbj_val), (0, len(str(ddbj_val))))
        rows.append({"field": field, "ChIP-Atlas": chip_val, "DDBJ RNA-Seq": ddbj_val})
    return pd.DataFrame(rows, columns=["field", "ChIP-Atlas", "DDBJ RNA-Seq"])


def show_overlap_example(organism: str, chip_prefix: str, ddbj_prefix: str) -> None:
    overlap = accession_sets[chip_prefix] & accession_sets[ddbj_prefix]
    accession = min(overlap)

    chip_category = next(c for c in CATEGORIES if c.dataset_prefix == chip_prefix)
    ddbj_category = next(c for c in CATEGORIES if c.dataset_prefix == ddbj_prefix)

    chip_record = find_bs_entry_in_files(category_input_files(chip_category), accession)
    ddbj_record = find_bs_entry_in_files(category_input_files(ddbj_category), accession)

    md(f"### {organism}: accession `{accession}`, in both ChIP-Atlas and DDBJ RNA-Seq")
    display_with_nested_tables(merged_overlap_table(chip_record, ddbj_record))


for organism, chip_prefix, ddbj_prefix in [
    ("Homo sapiens", "chipatlas_hg38", "rnaseq_human"),
    ("Mus musculus", "chipatlas_mm10", "rnaseq_mouse"),
]:
    show_overlap_example(organism, chip_prefix, ddbj_prefix)

### Homo sapiens: accession `SAMD00005178`, in both ChIP-Atlas and DDBJ RNA-Seq

field,ChIP-Atlas,DDBJ RNA-Seq
accession,SAMD00005178,SAMD00005178
internal_id,,
organism,Homo sapiens,Homo sapiens
taxonomy_id,9606,9606
title,10589-108D4,10589-108D4
sample_name,DRS007724,DRS007724
comment,,
access,public,public
submission_date,,
publication_date,2014-03-28T00:00:00+09:00,2014-03-28T00:00:00+09:00


### Mus musculus: accession `SAMD00004148`, in both ChIP-Atlas and DDBJ RNA-Seq

field,ChIP-Atlas,DDBJ RNA-Seq
accession,SAMD00004148,SAMD00004148
internal_id,,
organism,Mus musculus,Mus musculus
taxonomy_id,10090,10090
title,ZHBTc4 mouse ES cells stably expressing Flag-MacroH2A variant,ZHBTc4 mouse ES cells stably expressing Flag-MacroH2A variant
sample_name,DRS011871,DRS011871
comment,,
access,public,public
submission_date,,
publication_date,2013-06-25T00:00:00+09:00,2013-06-25T00:00:00+09:00


## One paired example per category

For each category, the smallest run's first record: the raw input as stored, and its matching curated entry, joined by `accession`. The header on each names the category and organism explicitly.

Each example ends with a deterministic trace-back check: for every non-null extracted field, does that value appear anywhere in the raw record -- checked first against every `Attributes` entry (in list order, including each attribute's own NAME, not just its content); only if none of them match at all is `Title`/`Description.Comment.Paragraph` checked as a fallback. Within a group, tiers are tried in order -- exact, case-insensitive, normalized (parenthetical/punctuation-insensitive), ontology synonym, fuzzy (spelling-tolerant, long words only) -- see `MATCH_STRATEGIES` and the "full crate" section below for what each one means. When a value matches more than one attribute, all of those matches are shown, not just the first. `matches` shows where it was found and the matched text in **bold**.


In [9]:
# PICK ONE (INPUT FILE, RESULT FILE) PER CATEGORY -- SMALLEST RUN, FOR A QUICK READ
def pick_example_run(category: Category) -> pd.Series:
    subset = run_index[(run_index["dataset"].str.startswith(category.dataset_prefix)) & (run_index["status"] == "completed")]
    return subset.loc[subset["total_input_entries"].idxmin()]


def show_paired_example(category: Category) -> None:
    run = pick_example_run(category)
    input_path = CRATE_INPUTS_DIR / run["dataset"] / run["input_file"]
    result_path = CRATE_RESULTS_DIR / run["result_file"]

    raw_record = next(iter_bs_entries(input_path))
    accession = get_accession(raw_record)

    select_result = load_select_result(result_path)
    curated_entry = find_entry(select_result, accession)
    ontology_indexes = get_ontology_indexes(run["select_config"])

    md(f"### {category.name} -- {category.organism} (`{run['run_name']}`, accession `{accession}`)")
    show_raw_record(raw_record, "Raw input")
    show_curated_record(curated_entry, "Curated output")

    md("**Tracing extracted values back to the raw input:**")
    trace_back_example = verify_extracted_against_raw(raw_record, curated_entry["extract"]["extracted"], curated_entry["results"], ontology_indexes)
    display_with_nested_tables(trace_back_example[["target_field", "target_value", "strategy", "matches", "n_matches", "assigned_term_id", "assigned_term_label"]])


for category in CATEGORIES:
    show_paired_example(category)

### DDBJ RNA-Seq -- Homo sapiens (`rnaseq_human_past_2014-09`, accession `SAMN01731102`)

(Live-verified: this table format reproduces every value in the raw JSON: **False**.)

**Raw input:**

**Curated output:**

extracted,results
field,value
cell_line,
cell_type,human umbilical vein endothelial cell
chip_antigen,
disease,
drug,
knockdown_gene,
knockout_gene,
overexpressed_gene,
tissue,


...

### ChIP-Atlas -- Mus musculus (`chipatlas_mm10_part5`, accession `SAMD00015968`)

**Raw input:**

**Curated output:**

extracted,results
field,value
cell_line,
cell_type,
chip_antigen,
disease,
drug,
knockdown_gene,
knockout_gene,
overexpressed_gene,MyoD
tissue,


**Tracing extracted values back to the raw input:**